In [27]:
#import packages
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

cases_df = pd.read_csv("weekly AFR cases by country as of 19 January 2025(in).csv")
print(cases_df.head())

  country iso3 week_end_date  total_confirmed_cases  total_suspected_cases  \
0  Angola  AGO    12/29/2024                      4                      0   
1  Angola  AGO    12/22/2024                      4                      0   
2  Angola  AGO    12/15/2024                      4                      0   
3  Angola  AGO     12/8/2024                      4                      0   
4  Angola  AGO     12/1/2024                      3                      0   

   total_deaths  total_suspected_deaths  new_confirmed_cases  \
0             0                       0                    0   
1             0                       0                    0   
2             0                       0                    0   
3             0                       0                    1   
4             0                       0                    1   

   new_suspected_cases  new_deaths  new_suspected_deaths  
0                    0           0                     0  
1                    0      

In [30]:
#cleaning cases data
cases_df['week_end_date'] = pd.to_datetime(cases_df['week_end_date'])

# Create lag feature for new weekly cases
cases_df['previous_week_cases'] = cases_df.groupby('country', group_keys=False)['total_confirmed_cases'].shift(1)

# Update column safely without using inplace=True
cases_df['previous_week_cases'] = cases_df['previous_week_cases'].fillna(0)

# Select relevant columns
cases_df_model = cases_df[['country', 'week_end_date', 'total_confirmed_cases', 'previous_week_cases']]
cases_df = cases_df.sort_values(by=['country', 'week_end_date'], ascending=[True, False])
cases_df_model.iloc[170:191]

,country,week_end_date,total_confirmed_cases,previous_week_cases
183,Burundi,2024-09-01,328,231.0
182,Burundi,2024-09-08,385,328.0
181,Burundi,2024-09-15,564,385.0
180,Burundi,2024-09-22,696,564.0
179,Burundi,2024-09-29,853,696.0
178,Burundi,2024-10-06,987,853.0
177,Burundi,2024-10-13,1170,987.0
176,Burundi,2024-10-20,1287,1170.0
175,Burundi,2024-10-27,1509,1287.0
174,Burundi,2024-11-03,1718,1509.0


In [52]:
healthcare_df = pd.read_csv("Healthcare expenditure data as percentage of GDP (%).csv")
healthcare_df.head()

,IndicatorCode,Indicator,ValueType,ParentLocationCode,ParentLocation,Location type,SpatialDimValueCode,Location,Period type,Period,...,FactValueUoM,FactValueNumericLowPrefix,FactValueNumericLow,FactValueNumericHighPrefix,FactValueNumericHigh,Value,FactValueTranslationID,FactComments,Language,DateModified
0,GHED_CHEGDP_SHA2011,Current health expenditure (CHE) as percentage...,numeric,WPR,Western Pacific,Country,BRN,Brunei Darussalam,Year,2022,...,NaN,NaN,NaN,NaN,NaN,1.82,NaN,NaN,EN,2024-12-10T00:00:00.000Z
1,GHED_CHEGDP_SHA2011,Current health expenditure (CHE) as percentage...,numeric,WPR,Western Pacific,Country,NZL,New Zealand,Year,2022,...,NaN,NaN,NaN,NaN,NaN,10.03,NaN,NaN,EN,2024-12-10T00:00:00.000Z
2,GHED_CHEGDP_SHA2011,Current health expenditure (CHE) as percentage...,numeric,AMR,Americas,Country,CHL,Chile,Year,2022,...,NaN,NaN,NaN,NaN,NaN,10.06,NaN,NaN,EN,2024-12-10T00:00:00.000Z
3,GHED_CHEGDP_SHA2011,Current health expenditure (CHE) as percentage...,numeric,EUR,Europe,Country,NLD,Netherlands (Kingdom of the),Year,2022,...,NaN,NaN,NaN,NaN,NaN,10.10,NaN,NaN,EN,2024-12-10T00:00:00.000Z
4,GHED_CHEGDP_SHA2011,Current health expenditure (CHE) as percentage...,numeric,WPR,Western Pacific,Country,FSM,Micronesia (Federated States of),Year,2022,...,NaN,NaN,NaN,NaN,NaN,10.34,NaN,NaN,EN,2024-12-10T00:00:00.000Z


In [38]:
#testing cases data

# Step 1: Have a data frame with relevant factors
df = cases_df_model.copy()

# Convert week_end_date to numerical format
df['week_end_date_num'] = df['week_end_date'].map(pd.Timestamp.toordinal)

# Step 2: Define features (x) and target (y)
x = df[['week_end_date_num', 'previous_week_cases']]
y = df['total_confirmed_cases']

# Step 3: Split data into training (80%) and testing (20%)
n = 10
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)

# Step 4: Train the Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=n)
rf.fit(x_train, y_train)

# Step 5: Compute Feature Importance
feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Step 6: Make Predictions on Test Data
y_pred = rf.predict(x_test)

# Step 7: Evaluate Model
mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae:.2f}')

# Step 8: Predict Future (e.g. 2025) Monkeypox Cases
new_data = pd.DataFrame({
    'week_end_date_num': [pd.Timestamp('2025-01-19').toordinal()],
    'previous_week_cases': [3116]
})

prediction = rf.predict(new_data)
print(f'Predicted Monkeypox Cases for 2025-01-19: {prediction[0]:.0f}')

Feature Importance:
               Feature  Importance
1  previous_week_cases    0.968429
0    week_end_date_num    0.031571
Mean Absolute Error: 8.44
Predicted Monkeypox Cases for 2025-01-19: 3461


In [ ]:
# Step 1: Have a data frame with data of the different factors we want to look at
# e.g. df = pd.DataFrame()

# (Step 1.5: Could create lag features for previous year, for the model to use previous year data as another factor)
# e.g. df['Lag_Healthcare_Expenditure'] = df['Healthcare_Expenditure'].shift(1)

# Step 2: Define factors (x) and target (y)
# e.g. x = df[['Year', 'Lag_Healthcare_Expenditure', 'Population_Density']]
#      y = df['Monkeypox_Cases']

# Step 4: Split data into training and testing (e.g. 80% train, 20% test)
# n = 10
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)
# random state to make sure results are repeatable

# Step 5: Train the Random Forest Model
# e.g. rf = RandomForestRegressor(n_estimators=100, random_state=n)
# rf.fit(x_train, y_train)

# Step 6: Compute Feature Importance
# feature_importance = pd.DataFrame({
#     'Feature': x.columns,
#     'Importance': rf.feature_importances_
# }).sort_values(by='Importance', ascending=False)

# Display Feature Importance
# print("Feature Importance:")
# print(feature_importance)

# Step 7: Make Predictions
# y_pred = rf.predict(x_test)

# Step 8: Evaluate the Model
# mae = mean_absolute_error(y_test, y_pred)
# print(f'Mean Absolute Error: {mae:.2f}')

# Step 9: Predict Future (e.g. 2025) Monkeypox Cases
# e.g. new_data = pd.DataFrame({
#     'Year': [2025],
#     'Lag_Healthcare_Expenditure': [5000],  # Previous year's value (2024)
#     'Lag_Population_Density': [150],  # Previous year's value (2024)
# })

# prediction_2025 = rf.predict(new_data)
# print(f'Predicted Monkeypox Cases for 2025: {prediction_2025[0]:.0f}')